# 🖌️🎮 Air Motion Studio: Canvas, Air XO Arena & Leaderboard - Google Colab

This notebook provides a complete interactive **Air Writing Canvas**, **Air XO Game**, **Leaderboard System**, and **Match History Tracking** inside Google Colab.

### 🖐️ Gesture Reference:
- ☝️ **1 Finger Up (Index)**: Draw Mode (Studio) / Target XO Grid Cell
- ⏱️ **0.8 Sec Dwell Hold**: Confirm Air XO Move
- ✌️ **2 Fingers Up (Index + Middle)**: Pointer / Hover Mode
- 🤟 **3 Fingers Up (Index + Middle + Ring)**: Clear Canvas / Reset Board
- 🖐️ **Open Palm**: Pause Mode

In [ ]:
import sys
import subprocess

def setup_environment():
    packages = ['opencv-python', 'mediapipe', 'numpy', 'Pillow']
    for package in packages:
        try:
            __import__(package.replace('-', '_'))
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', package], check=True)

setup_environment()
print("Environment initialized successfully!")

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import base64
from PIL import Image
import io
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js

class ColabAirMotionStudio:
    def __init__(self, width=640, height=480):
        self.width = width
        self.height = height
        self.canvas = np.zeros((height, width, 3), dtype=np.uint8)
        self.prev_x, self.prev_y = 0, 0
        self.draw_color = (255, 0, 128)
        self.brush_thickness = 6
        
        self.mp_hands = mp.solutions.hands
        self.mp_draw = mp.solutions.drawing_utils
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7,
            min_tracking_confidence=0.7
        )

    def process_frame(self, frame_img):
        h, w, c = frame_img.shape
        if self.canvas.shape[:2] != (h, w):
            self.canvas = np.zeros((h, w, 3), dtype=np.uint8)

        frame = cv2.flip(frame_img, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.hands.process(rgb_frame)
        gesture_status = "Neutral"

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                self.mp_draw.draw_landmarks(frame, hand_landmarks, self.mp_hands.HAND_CONNECTIONS)
                lm = hand_landmarks.landmark
                idx_x, idx_y = int(lm[8].x * w), int(lm[8].y * h)

                idx_up = lm[8].y < lm[6].y
                mid_up = lm[12].y < lm[10].y
                ring_up = lm[16].y < lm[14].y
                pinky_up = lm[20].y < lm[18].y

                if idx_up and not mid_up and not ring_up and not pinky_up:
                    gesture_status = "Drawing"
                    cv2.circle(frame, (idx_x, idx_y), self.brush_thickness + 2, self.draw_color, -1)
                    if self.prev_x == 0 and self.prev_y == 0:
                        self.prev_x, self.prev_y = idx_x, idx_y
                    cv2.line(self.canvas, (self.prev_x, self.prev_y), (idx_x, idx_y), self.draw_color, self.brush_thickness)
                    self.prev_x, self.prev_y = idx_x, idx_y
                elif idx_up and mid_up and not ring_up and not pinky_up:
                    gesture_status = "Hovering"
                    self.prev_x, self.prev_y = 0, 0
                    cv2.circle(frame, (idx_x, idx_y), 10, (0, 255, 255), 2)
                elif idx_up and mid_up and ring_up and not pinky_up:
                    gesture_status = "Canvas Cleared"
                    self.canvas.fill(0)
                    self.prev_x, self.prev_y = 0, 0
                else:
                    gesture_status = "Paused"
                    self.prev_x, self.prev_y = 0, 0
        else:
            self.prev_x, self.prev_y = 0, 0

        gray = cv2.cvtColor(self.canvas, cv2.COLOR_BGR2GRAY)
        _, inv = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY_INV)
        inv = cv2.cvtColor(inv, cv2.COLOR_GRAY2BGR)
        bg = cv2.bitwise_and(frame, inv)
        merged = cv2.add(bg, self.canvas)

        cv2.putText(merged, f"Gesture: {gesture_status}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        return merged

print("ColabAirMotionStudio Loaded.")

In [ ]:
studio = ColabAirMotionStudio()
print("Air Motion Studio Ready!")